# Mobile Money Agent Network Data Audit and Cleaning
**Author:** Asman Makero Musa · PLP Data Analytics Cohort 7

## Task 1: Data Quality Audit Across Six Dimensions

In [1]:
import pandas as pd
import numpy as np

raw_txns = pd.read_csv('agent_transactions.csv', keep_default_na=False)
raw_agents = pd.read_csv('agents.csv')

audit_data = [
    {'issue': 'Duplicate txn_id', 'dimension': 'Uniqueness', 'column': 'txn_id', 'rows_affected': int(raw_txns.duplicated(subset=['txn_id']).sum())},
    {'issue': 'Casing/whitespace variance', 'dimension': 'Consistency', 'column': 'txn_type', 'rows_affected': int((~raw_txns['txn_type'].isin(['Deposit', 'Withdrawal'])).sum())},
    {'issue': 'Missing/empty amounts', 'dimension': 'Completeness', 'column': 'amount', 'rows_affected': int(raw_txns['amount'].isin(['', 'N/A', 'nan']).sum())},
    {'issue': 'Slash date (YYYY/MM/DD)', 'dimension': 'Validity', 'column': 'txn_date', 'rows_affected': int(raw_txns['txn_date'].str.contains('/').sum())},
    {'issue': 'Spike values (KES 250k)', 'dimension': 'Accuracy', 'column': 'amount', 'rows_affected': int((raw_txns['amount'] == '250000').sum())},
    {'issue': 'Unregistered agent (AG099)', 'dimension': 'Validity', 'column': 'agent_id', 'rows_affected': int((~raw_txns['agent_id'].isin(raw_agents['agent_id'])).sum())}
]
audit_df = pd.DataFrame(audit_data)
print(audit_df.to_string(index=False))

                     issue    dimension   column  rows_affected
          Duplicate txn_id   Uniqueness   txn_id             12
Casing/whitespace variance  Consistency txn_type            384
     Missing/empty amounts Completeness   amount             22
   Slash date (YYYY/MM/DD)     Validity txn_date              5
   Spike values (KES 250k)     Accuracy   amount              5
Unregistered agent (AG099)     Validity agent_id              8


## Tasks 2–7: Execute Cleaning Pipeline from Modular Script

In [2]:
from cleaning_pipeline import clean_agent_data

raw_txns_load = pd.read_csv('agent_transactions.csv')
clean_txns, cleaning_log = clean_agent_data(raw_txns_load, raw_agents)

print(f'Before shape: {raw_txns_load.shape} | After shape: {clean_txns.shape}')
print('=== Cleaning Log ===')
for msg in cleaning_log:
    print(f'- {msg}')

Before shape: (612, 5) | After shape: (600, 8)
=== Cleaning Log ===
- Removed 12 duplicate transactions based on txn_id.
- Standardized txn_type into 'Deposit' and 'Withdrawal'.
- Imputed 20 missing/invalid amounts with median (KES 1,776).
- Parsed and standardized txn_date to ISO datetime format.
- Flagged 53 high-value outliers exceeding upper fence KES 5,513.75.
- Merged with agent register; 7 transactions with unknown agents assigned to 'Unknown' county.


## Post-Merge Validation

In [3]:
pre_merge_count = len(clean_txns)
unknown_agent_count = int((clean_txns['agent_id'] == 'AG099').sum())
print(f'Post-merge row count: {len(clean_txns)} | Matches pre-merge: {pre_merge_count == len(clean_txns)}')
print(f'Unknown agent (AG099) count: {unknown_agent_count} -> Assigned county: Unknown')
print('County distribution:', clean_txns['county'].value_counts().to_dict())

Post-merge row count: 600 | Matches pre-merge: True
Unknown agent (AG099) count: 7 -> Assigned county: Unknown
County distribution: {'Nairobi': 180, 'Mombasa': 133, 'Kisumu': 111, 'Nakuru': 99, 'Kiambu': 70, 'Unknown': 7}


## Task 8(a): County-Level Transaction Summary by Transaction Type

In [4]:
county_summary = clean_txns.groupby(['county', 'txn_type']).agg(total_value=('amount', 'sum'), txn_count=('txn_id', 'count'), avg_value=('amount', 'mean')).round(2).reset_index()
print(county_summary.to_string(index=False))

 county   txn_type  total_value  txn_count  avg_value
 Kiambu    Deposit     334737.0         43    7784.58
 Kiambu Withdrawal      77050.5         27    2853.72
 Kisumu    Deposit     436420.5         74    5897.57
 Kisumu Withdrawal      76374.5         37    2064.18
Mombasa    Deposit     148986.0         72    2069.25
Mombasa Withdrawal     368592.0         61    6042.49
Nairobi    Deposit     543590.0        118    4606.69
Nairobi Withdrawal     162036.0         62    2613.48
 Nakuru    Deposit     184890.0         60    3081.50
 Nakuru Withdrawal     328594.5         39    8425.50
Unknown    Deposit      25279.0          6    4213.17
Unknown Withdrawal       1877.0          1    1877.00


## Task 8(b): Weekly Transaction Value by County Pivot Table

In [5]:
clean_txns['week'] = clean_txns['txn_date'].dt.to_period('W').astype(str)
weekly_pivot = pd.pivot_table(clean_txns, index='county', columns='week', values='amount', aggfunc='sum', fill_value=0)
print('Weekly Pivot (First 4 Weeks):\n', weekly_pivot.iloc[:, :4].astype(int))
print('County Totals (KES):\n', weekly_pivot.sum(axis=1).astype(int).to_dict())

Weekly Pivot (First 4 Weeks):
 week     2026-02-23/2026-03-01  ...  2026-03-16/2026-03-22
county                          ...                       
Kiambu                       0  ...                  22054
Kisumu                    1893  ...                  28125
Mombasa                   6759  ...                  25144
Nairobi                   7046  ...                  43107
Nakuru                    2743  ...                  39795
Unknown                      0  ...                   1639

[6 rows x 4 columns]
County Totals (KES):
 {'Kiambu': 411787, 'Kisumu': 512795, 'Mombasa': 517578, 'Nairobi': 705626, 'Nakuru': 513484, 'Unknown': 27156}


In [6]:
clean_txns.to_csv('outputs/agent_transactions_clean.csv', index=False)
print('Saved outputs/agent_transactions_clean.csv | Shape:', clean_txns.shape)
print('Columns:', ['amount_imputed', 'is_outlier'] + [c for c in clean_txns.columns if c not in ['amount_imputed', 'is_outlier']])

Saved outputs/agent_transactions_clean.csv | Shape: (600, 9)
Columns: ['amount_imputed', 'is_outlier', 'txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'county', 'week']


# Data Quality Report

### Six-Dimension Quality Audit
- **Uniqueness:** 12 duplicates purged (600 clean transactions from 612 raw).
- **Consistency:** `txn_type` variants normalized to canonical `'Deposit'` and `'Withdrawal'`.
- **Validity:** 5 slash dates converted to ISO. 7 unregistered AG099 txns mapped to county `'Unknown'`.
- **Completeness:** 20 missing amounts imputed with median (**KES 1,776**) and flagged in `amount_imputed`.
- **Accuracy & Outliers:** 53 outliers (>KES 5,513.75 IQR fence) flagged in `is_outlier`, isolating 5 artificial KES 250k spikes for AML review.
- **Timeliness:** Records span Mar 1 to Apr 30, 2026, providing complete operational coverage.

### Business Controls & Privacy (KDPA 2019)
- **Controls:** Enforce DB primary keys on `txn_id`, dropdown validation for `txn_type`, and real-time AML alerts above KES 100k.
- **Privacy Compliance:** Under Kenya Data Protection Act 2019, phone numbers and customer identifiers must be pseudonymised via salted SHA-256 and masked (`0712****78`) in analytics views. Unaggregated raw records must remain protected with strict access and retention limits.